# Sortitude — Edge AI Waste Classifier
### Google Coral Dev Board Deployment Pipeline

**Dataset:** TrashNet — 6 classes: cardboard, glass, metal, paper, plastic, trash  
**Model:** MobileNetV2 transfer learning  
**Output:** 3 TFLite variants → Float32 · Int8 · Pruned+Quant → Edge TPU compiled

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q tensorflow-model-optimization gdown

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os, zipfile
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import tensorflow_model_optimization as tfmot
import gdown

print('TF version :', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Cell 3: Mount Google Drive & set paths ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH    = '/content/drive/MyDrive/Colab_Notebooks'
DATASET_PATH = f'{BASE_PATH}/TrashNet/dataset-resized'
MODELS_PATH  = f'{BASE_PATH}/models'

os.makedirs(f'{BASE_PATH}/TrashNet', exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)
print('Paths ready.')

In [ ]:
# ── Cell 4: Download TrashNet dataset ─────────────────────────────────────────
# TrashNet by Gary Thung & Mindy Yang (Stanford)
# ~3.5 GB, 2527 images across 6 classes

ZIP_PATH = f'{BASE_PATH}/TrashNet/dataset-resized.zip'

if not os.path.exists(DATASET_PATH):
    print('Downloading TrashNet...')
    # Public Google Drive mirror of the dataset
    gdown.download(
        'https://drive.google.com/uc?id=1iD0d7EMkknFrECMNpZFLzb5v_AO5kIZq',
        ZIP_PATH, quiet=False
    )
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(f'{BASE_PATH}/TrashNet/')
    os.remove(ZIP_PATH)
    print('Done!')
else:
    print('Dataset already present, skipping download.')

classes = sorted(os.listdir(DATASET_PATH))
print(f'Classes ({len(classes)}):', classes)
for c in classes:
    n = len(os.listdir(f'{DATASET_PATH}/{c}'))
    print(f'  {c}: {n} images')

In [ ]:
# ── Cell 5: Data pipeline ─────────────────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42
NUM_CLASSES = 6

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
print('Class names:', class_names)

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# ── Cell 6: Preview sample images ─────────────────────────────────────────────
plt.figure(figsize=(12, 6))
for images, labels in train_ds.take(1):
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[np.argmax(labels[i])])
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Build MobileNetV2 transfer learning model ─────────────────────────
def build_model(num_classes=NUM_CLASSES):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False   # freeze for Phase 1

    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    # MobileNetV2 expects [-1, 1] range
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()

In [ ]:
# ── Cell 8: Phase 1 training — frozen base, train head only ───────────────────
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    f'{MODELS_PATH}/best_model.keras',
    monitor='val_accuracy', save_best_only=True, verbose=1
)
early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True
)

history = model.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=[checkpoint_cb, early_stop_cb]
)

In [ ]:
# ── Cell 9: Phase 2 fine-tuning — unfreeze top layers ─────────────────────────
base_model = model.layers[3]   # MobileNetV2 base
base_model.trainable = True

# Unfreeze only the top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=[checkpoint_cb, early_stop_cb]
)

In [ ]:
# ── Cell 10: Plot training history ────────────────────────────────────────────
def plot_history(h1, h2=None):
    acc  = h1.history['accuracy']
    vacc = h1.history['val_accuracy']
    if h2:
        acc  += h2.history['accuracy']
        vacc += h2.history['val_accuracy']

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(acc,  label='Train')
    plt.plot(vacc, label='Val')
    if h2:
        plt.axvline(len(h1.history['accuracy']) - 1, color='gray', linestyle='--', label='Fine-tune start')
    plt.title('Accuracy'); plt.legend()

    loss  = h1.history['loss']
    vloss = h1.history['val_loss']
    if h2:
        loss  += h2.history['loss']
        vloss += h2.history['val_loss']
    plt.subplot(1, 2, 2)
    plt.plot(loss,  label='Train')
    plt.plot(vloss, label='Val')
    plt.title('Loss'); plt.legend()
    plt.tight_layout()
    plt.show()

plot_history(history, history_fine)

loss, acc = model.evaluate(val_ds)
print(f'\nFinal val accuracy: {acc*100:.2f}%')

In [ ]:
# ── Cell 11: Float32 TFLite conversion ────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float = converter.convert()

FLOAT32_PATH = f'{MODELS_PATH}/model_float32.tflite'
with open(FLOAT32_PATH, 'wb') as f:
    f.write(tflite_float)

print(f'Float32 size: {len(tflite_float) / 1e6:.2f} MB')

In [ ]:
# ── Cell 12: Int8 quantized TFLite (Edge TPU-ready) ───────────────────────────
def representative_dataset():
    for images, _ in val_ds.unbatch().batch(1).take(200):
        yield [images]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_int8 = converter.convert()

INT8_PATH = f'{MODELS_PATH}/model_int8.tflite'
with open(INT8_PATH, 'wb') as f:
    f.write(tflite_int8)

print(f'Int8 size: {len(tflite_int8) / 1e6:.2f} MB')

In [ ]:
# ── Cell 13: Pruned + Quantized TFLite ────────────────────────────────────────
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

num_steps = np.ceil(len(train_ds) * 5).astype(np.int32)

pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.50,
        begin_step=0,
        end_step=num_steps
    )
}

pruned_model = prune_low_magnitude(model, **pruning_params)
pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

pruned_model.fit(
    train_ds, epochs=5, validation_data=val_ds,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()]
)

# Strip pruning wrappers
final_pruned = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Quantize the pruned model
converter = tf.lite.TFLiteConverter.from_keras_model(final_pruned)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_pruned_q = converter.convert()

PRUNED_PATH = f'{MODELS_PATH}/model_pruned_quant.tflite'
with open(PRUNED_PATH, 'wb') as f:
    f.write(tflite_pruned_q)

print(f'Pruned+Quant size: {len(tflite_pruned_q) / 1e6:.2f} MB')

In [ ]:
# ── Cell 14: Compile for Edge TPU ─────────────────────────────────────────────
# Install the Edge TPU compiler
!curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo gpg --dearmor -o /usr/share/keyrings/coral-edgetpu.gpg
!echo 'deb [signed-by=/usr/share/keyrings/coral-edgetpu.gpg] https://packages.cloud.google.com/apt coral-edgetpu-stable main' | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
!sudo apt-get update -q && sudo apt-get install -y edgetpu-compiler

# Compile the int8 model
!edgetpu_compiler {INT8_PATH} -o {MODELS_PATH}/

# Check output — should report % ops mapped to Edge TPU
!ls -lh {MODELS_PATH}/

In [ ]:
# ── Cell 15: Benchmark all 3 TFLite variants ──────────────────────────────────
def eval_tflite(tflite_path, ds, num_batches=20):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    inp  = interpreter.get_input_details()[0]
    out  = interpreter.get_output_details()[0]
    is_int8 = inp['dtype'] == np.int8

    correct, total = 0, 0
    import time
    times = []

    for images, labels in ds.unbatch().batch(1).take(num_batches * 32):
        img = images.numpy()
        if is_int8:
            scale, zero_point = inp['quantization']
            img = (img / scale + zero_point).astype(np.int8)

        interpreter.set_tensor(inp['index'], img)
        t0 = time.perf_counter()
        interpreter.invoke()
        times.append(time.perf_counter() - t0)

        pred = interpreter.get_tensor(out['index'])
        if is_int8:
            scale_o, zp_o = out['quantization']
            pred = (pred.astype(np.float32) - zp_o) * scale_o

        correct += int(np.argmax(pred) == np.argmax(labels.numpy()))
        total   += 1

    avg_ms = np.mean(times) * 1000
    size_mb = os.path.getsize(tflite_path) / 1e6
    return size_mb, correct / total * 100, avg_ms

print(f'{"Model":<25} {"Size (MB)":>10} {"Val Acc %":>10} {"Avg ms":>10}')
print('-' * 60)

for name, path in [
    ('Float32',        FLOAT32_PATH),
    ('Int8 Quantized', INT8_PATH),
    ('Pruned+Quant',   PRUNED_PATH),
]:
    size, acc, ms = eval_tflite(path, val_ds)
    print(f'{name:<25} {size:>10.2f} {acc:>10.2f} {ms:>10.2f}')

## Next Steps — Deploy to Coral

Copy these files to the Coral Dev Board:
```
models/model_int8_edgetpu.tflite   ← compiled model
labels.txt                          ← one class name per line
demo.py                             ← inference script
```

Run on board:
```bash
python3 demo.py --model model_int8_edgetpu.tflite --labels labels.txt
```